# 开上你的梦幻赛道：用 AI 分镜跑完一圈（本地版）

我们手里有一张自己设计的赛道俯视图。任务：生成一段车手第一视角的视频，把这条赛道完整跑一圈。

| 章节 | 内容 | 消耗额度 |
|---|---|---|
| 1 | 认识赛道地图，抛出任务 | 无 |
| 2 | 一个提示词直接生成：15 秒、跑不完、不稳定 | 每次 1 段视频（可选） |
| 3 | 分镜：把一圈拆成 7 个镜头 | 无 |
| 4 | 关键帧：每个镜头交界处一张车手视角图，对照分镜检查 | 8 张图起，每改一次多 1 张（可直接用自带素材） |
| 5 | 逐段拍镜头：首尾帧生成视频 | 最多 14 段视频（可直接用自带素材） |
| 6 | 拼接成片，找出问题 | 无 |
| 7 | 用分镜修好 T7–T8 减速弯 | 2 张关键帧，3 段视频 × 2 候选（可直接用自带素材） |
| 8 | 总结和练习 | 无 |

这个 notebook 只负责准备环境、启动 ComfyUI、准备素材和拼接成片；生成图片和视频都在浏览器里的 ComfyUI 完成。
每个需要额度的步骤，本项目都提供了当时实际生成的结果，可以不花额度跟完全程。

### 怎么用这个 notebook

- 从上往下逐格运行：点中一格，按 `Shift+Enter`。左边显示 `[*]` 表示还在运行，变成数字才算完成，输出显示在这一格下面。
- 两条路线，可以每章自由选：
  - **路线 A，不花额度**：所有单元格按顺序运行，自带素材会补上关键帧和视频。
  - **路线 B，自己生成**：遇到“在浏览器里”的步骤，先去 ComfyUI 生成，再运行“挑选”单元格（视频要先填上你选中的候选编号）；没生成的部分仍由自带素材补上。
- 重启 kernel 或第二天继续时，从第 0 节重新运行即可，已经完成的步骤会自动跳过。

## 0. 准备环境

本地版使用本项目目录的上一级 ComfyUI（`dream-circuit-lap` 放在 ComfyUI 目录里），也可以用环境变量 `COMFYUI_DIR` 指定。
请用 ComfyUI 所在的 Python 环境作为这个 notebook 的内核，下面启动 ComfyUI 时用的就是这个解释器。

In [ ]:
import os, sys, shutil, socket, subprocess, time, urllib.request
from getpass import getpass
from pathlib import Path

PROJECT = Path.cwd().parent
COMFY = Path(os.environ.get("COMFYUI_DIR", PROJECT.parent)).resolve()
PY = sys.executable
assert (COMFY / "main.py").is_file(), f"ComfyUI not found at {COMFY}, set COMFYUI_DIR"
os.environ["COMFYUI_DIR"] = str(COMFY)
print("project:", PROJECT)
print("ComfyUI:", COMFY)
print("python :", PY)

In [ ]:
%pip install -q -r ../requirements.txt

### 安装两个云端节点

[ComfyUI-AMD-Qwen-Image-2.1](https://github.com/wangxunx/ComfyUI-AMD-Qwen-Image-2.1) 生成关键帧，[ComfyUI-AMD-MiniMaxH3](https://github.com/wangxunx/ComfyUI-AMD-MiniMaxH3) 生成视频。
两者都在云端推理，本机不需要 GPU。已经装过的会跳过。

In [ ]:
for repo in ("ComfyUI-AMD-Qwen-Image-2.1", "ComfyUI-AMD-MiniMaxH3"):
    if (COMFY / "custom_nodes" / repo).is_dir():
        print("installed:", repo)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/wangxunx/{repo}.git", str(COMFY / "custom_nodes" / repo)], check=True)
        print("cloned:", repo)

### 配置 H3 key

到 https://h3.oneclickamd.ai 注册拿到 key。key 写进 ComfyUI 的 `user/h3_gateway_api_key.txt`，两个节点都从这里读，不会写进工作流或视频元数据。
运行下一格时，输出区会出现一个输入框：粘贴 key，按回车。

In [ ]:
key_file = COMFY / "user" / "h3_gateway_api_key.txt"
if os.environ.get("H3_GATEWAY_API_KEY") or key_file.is_file():
    print("H3 key already configured")
else:
    key_file.parent.mkdir(parents=True, exist_ok=True)
    key_file.write_text(getpass("H3 key: ").strip())
    print("saved to", key_file)

### 启动 ComfyUI

ComfyUI 在后台启动，第一次大约要等十几秒，日志在 ComfyUI 的 `user/comfyui_dream_circuit.log`。
8188 端口已经有 ComfyUI 在运行时直接使用它；如果它是在安装节点之前启动的，请先重启，否则看不到新节点。

Jupyter 跑在远程机器上（例如通过 Cursor / VS Code 的 Remote SSH）时，浏览器打不开 http://127.0.0.1:8188 的话，在 **Ports（端口）** 面板里转发 8188 端口。

In [ ]:
def comfy_up():
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", 8188)) == 0

if comfy_up():
    print("ComfyUI is already running")
else:
    log = open(COMFY / "user" / "comfyui_dream_circuit.log", "w")
    subprocess.Popen([PY, "main.py", "--port", "8188", "--cpu"], cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        time.sleep(1)
        if comfy_up():
            break
    else:
        raise RuntimeError(f"ComfyUI did not start, see {log.name}")
print("open http://127.0.0.1:8188")

### 准备素材

下载历史生成结果（约 72 MB，带 sha256 校验，可以重复运行）；系统里没有中文字体时顺带下载 Noto Sans SC，拼接时的中文标题要用。
然后把各章工作流装进 ComfyUI：浏览器左侧 **工作流** 面板里会出现 `ch2_single_prompt`、`dream7_*`、`dream8_*`。

In [ ]:
!"{PY}" "{PROJECT}/scripts/fetch_media.py" --fonts
(COMFY / "user" / "default" / "workflows").mkdir(parents=True, exist_ok=True)
shutil.copy(PROJECT / "workflows" / "ch2_single_prompt.json", COMFY / "user" / "default" / "workflows")
shutil.copy(PROJECT / "assets" / "fictional_track_v1.png", COMFY / "input")
!"{PY}" "{PROJECT}/scripts/make_workflows.py" --install
!"{PY}" "{PROJECT}/scripts/make_workflows.py" --shots lap_8shots.json --install

### 第一次用 ComfyUI 先看这里

1. 浏览器打开 ComfyUI（本地版是 http://127.0.0.1:8188，云端版是上面打印的 `ComfyUI URL`）。第一次打开会弹出 Templates 模板窗口，点右上角 × 关掉。中间是画布，方框是节点，连线是数据的流向。
2. 按 `w` 或点左侧 **Workflows（工作流）**，列表里点名字就能打开工作流。看不到本项目的工作流时，刷新浏览器页面（F5）。
3. 本项目的每个工作流左边都有一个黄色的说明节点，写着这一步做什么、运行前要准备什么。滚轮缩放，按住空白处拖动画布，按 `.` 显示全部节点。
4. 点右上角蓝色的 **Run（运行）** 提交任务，旁边的 “N active” 是还在进行的任务数。生成在云端完成，视频通常要等几分钟。
5. 结果显示在 Save Image / Save Video 节点里，同时保存在 ComfyUI 的 `output/` 目录。
6. Load Image 节点被红框圈住、右上角提示 “Media input missing” 或 “errors found”：它要读的图片还不在 `input/`。先运行 notebook 里对应的复制或挑选单元格，再刷新浏览器页面（F5），红框就会消失。
7. 第 4–7 章的工作流不要改文件名前缀，后面的挑选和拼接按这些文件名找文件。

下面的 `play` 用相对路径引用视频，需要在 JupyterLab / JupyterHub 里打开这个 notebook，并且本项目目录在 Jupyter 的根目录之下；图片缩小后直接嵌入。

In [ ]:
from IPython.display import HTML, Markdown, display
from base64 import b64encode
from io import BytesIO
from PIL import Image
import json

MEDIA = PROJECT / "media"

def figure(inner, name):
    return f'<figure style="display:inline-block;margin:4px">{inner}<figcaption>{name}</figcaption></figure>'

def play(*paths, width=480):
    display(HTML("".join(figure(f'<video src="{os.path.relpath(p)}" width="{width}" controls muted></video>', Path(p).name) for p in paths)))

def show(*paths, width=360):
    tags = []
    for p in paths:
        im = Image.open(p).convert("RGB")
        im.thumbnail((width * 2, width * 2))
        buf = BytesIO()
        im.save(buf, "JPEG", quality=85)
        tags.append(figure(f'<img src="data:image/jpeg;base64,{b64encode(buf.getvalue()).decode()}" width="{width}">', Path(p).name))
    display(HTML("".join(tags)))

def copy_missing(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy(src, dst)
        print("copied", dst)

def pick(patterns, src_dir, dst):
    missing = []
    for name, pattern in patterns.items():
        found = sorted(Path(src_dir).glob(pattern), key=os.path.getmtime)
        if not found:
            missing.append(name)
            continue
        target = Path(str(dst).format(name))
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(found[-1], target)
        print(found[-1].name, "->", target)
    if missing:
        print("not generated yet:", " ".join(missing))

## 第 1 章：赛道地图和任务

5.1 km，11 个弯，逆时针。从起终点直道出发，依次是重刹弯 T1–T2、爬坡弯组 T3–T5、高速大半径弯 T6、短直道和减速弯 T7–T8、下坡弯 T9、最后赛段 T10–T11。

任务：根据这张图，生成一段车手第一视角的视频，体验跑完一圈。讲解提纲见 `docs/ch1_map.md`。

In [ ]:
show(PROJECT / "assets" / "fictional_track_v1.png", width=900)

## 第 2 章：一个提示词跑一圈

最直接的想法：把俯视图当首帧，用一个提示词描述 7 个赛道元素，让 MiniMax H3 生成。
在浏览器里打开 `ch2_single_prompt` 工作流就能试（每次 1 段视频额度）。下面是当时的 9 次尝试：

- 1–6 号一直在改提示词：强调逆时针、说明起点位置、要求中文配音。但视频都从地图飞进去，“第一视角”一直嵌在地图的图例和海拔剖面中间，15 秒只够跑一小段，弯道也和地图对不上。
- 5 号和 6 号的提示词完全相同，结果的结构几乎一样，只有细节不同：重跑一次救不了，结果是被输入决定的。
- 7–8 号把目标缩小为“只拍起终点直道到 T1”，画风却变成了卡通；9 号再加上“路况和两侧环境与俯视图一致”，终于得到全屏、写实的座舱视角。

两个启发：一次只拍一小段；首帧不该是俯视图，而应该是车手视角的画面。这就是分镜和关键帧的起点。

In [ ]:
play(MEDIA / "ch2_single_prompt_01.mp4", MEDIA / "ch2_single_prompt_05.mp4", MEDIA / "ch2_single_prompt_06.mp4", width=360)
play(MEDIA / "ch2_single_prompt_07.mp4", MEDIA / "ch2_single_prompt_09.mp4", width=360)

## 第 3 章：分镜

把一圈拆成 7 个镜头，每个镜头 5–8 秒。相邻镜头共用一张交界处的关键帧 K0–K7：第 i 个镜头以 K(i-1) 为首帧、K(i) 为尾帧，所以能接上。
分镜表在 `storyboards/lap_7shots.json`，生成的手册是 `docs/storyboard_dream7.md`。

In [ ]:
sb = json.load(open(PROJECT / "storyboards" / "lap_7shots.json", encoding="utf-8"))
rows = ["| 镜头 | 赛段 | 首帧 → 尾帧 | 时长 | 弯道 |", "|---|---|---|---|---|"]
rows += [f"| {s['id']} | {s['title']} | {s['first']} → {s['last']} | {s['duration']} 秒 | {s['turns']} |" for s in sb["shots"]]
display(Markdown("\n".join(rows)))

## 第 4 章：关键帧

关键帧是镜头之间的“合同”：视频从首帧出发、必须到达尾帧，尾帧里画了什么，模型就往哪里开。所以每张图都要对照分镜检查，最重要的一条是：前方的路往哪边弯，要和地图一致。

节点没有随机种子，提示词不变时重新生成也是同一张图。所以每张关键帧只生成 1 次；不对的时候，找出哪里和分镜不符，把它写进场景描述，再生成。下面是原项目里 K0 的 4 个版本，每一版都只改了一个问题：

1. 直道远端先左弯、再右弯、再上坡；地图上从起终点线只能看到 T1 一个左弯。场景改为直道笔直、尽头只有一个左弯、四周平坦。
2. 左弯离得太近，直道看起来只有两三百米；地图上约 1 km。场景改为直道一直延伸到消失点，左弯只在消失点处隐约可见。
3. 路对了，但起跑灯架远在红车后面，红灯只剩几个小点。场景改为灯架在车前约 20 米、占满画面上部。
4. 定稿。

7 镜头成片用的是第 1 版，第 6 章会看到它带来的问题。

In [ ]:
show(*[PROJECT / "examples" / "k0_versions" / f"K0_v{v}.jpg" for v in range(1, 5)], width=440)

在浏览器里：

1. 打开 `dream7_keyframes_pilot`，点 Run，先只生成 K0、K1，看画风和机位是否满意。
2. 打开 `dream7_keyframes`，点 Run 生成全部 8 张，每张一行。
3. 逐张对照第 3 章的分镜表和小地图检查：路往哪边弯、离弯道多远、灯架和看台这类元素在不在、机位和赛车是否一致。
4. 哪张不对，就改那个节点提示词开头的场景描述（`Photorealistic onboard` 之前的部分），再点 Run。ComfyUI 只会重新生成改过的那张，每次 1 个额度；ComfyUI 重启过的话会全部重新生成，可以先选中不用重做的 Save Image 节点按 Ctrl+M 静音。
5. 都满意后运行下一格，每张关键帧最新的那一版会复制到 ComfyUI 的 `input/dream7_K0.png` 到 `dream7_K7.png`。

路线 A 直接运行下面两格即可：挑选格找不到自己生成的图就什么都不做，再下一格用自带的关键帧补上缺的，不会覆盖你选好的文件。讲解提纲见 `docs/ch4_keyframes.md`。

In [ ]:
pick({f"K{i}": f"K{i}_*.png" for i in range(8)}, COMFY / "output" / "dream7", COMFY / "input" / "dream7_{}.png")

In [ ]:
for i in range(8):
    copy_missing(PROJECT / "examples" / "keyframes" / f"dream7_K{i}.png", COMFY / "input" / f"dream7_K{i}.png")
show(*[COMFY / "input" / f"dream7_K{i}.png" for i in range(8)], width=300)

## 第 5 章：逐段拍镜头

第 i 个镜头以 K(i-1) 为首帧、K(i) 为尾帧，交给 MiniMax H3 首尾帧节点生成，提示词只描述“怎么动”。

在浏览器里：

0. 先完成第 4 章（关键帧已经在 `input/` 里），再刷新一次浏览器页面，否则 Load Image 节点会报红。
1. 打开 `dream7_shot_01` 拍镜头 1（试拍），每次 Run 生成 2 个候选：上面一条是 `_t1`，下面一条是 `_t2`。看画风是否满意、开头和结尾是否贴近 K0 和 K1、方向和速度感是否对。
2. 满意后依次打开 `dream7_shot_02` 到 `dream7_shot_07` 并 Run。可以连续提交，任务会排队。
3. 把每个镜头选中的候选编号填进下一格的 `SHOT_TAKES` 并运行，选中的视频会复制为 `output/video/dream7/selected/shot_0X.mp4`。

挑视频时看：转向方向对不对、有没有切镜或倒车、车和机位有没有中途变化、首尾是否贴近关键帧。
路线 A 直接运行下面两格，自带的视频会补上缺的镜头，不会覆盖你选好的文件；下面播放的是自带素材里的镜头 1、2。讲解提纲见 `docs/ch5_shots.md`。

In [ ]:
SHOT_TAKES = {"shot_01": 1, "shot_02": 1, "shot_03": 1, "shot_04": 1, "shot_05": 1, "shot_06": 1, "shot_07": 1}
pick({s: f"{s}_t{t}_*" for s, t in SHOT_TAKES.items()}, COMFY / "output" / "video" / "dream7", COMFY / "output" / "video" / "dream7" / "selected" / "{}.mp4")

In [ ]:
for i in range(1, 8):
    copy_missing(MEDIA / f"dream7_shot_0{i}.mp4", COMFY / "output" / "video" / "dream7" / "selected" / f"shot_0{i}.mp4")
play(MEDIA / "dream7_shot_01.mp4", MEDIA / "dream7_shot_02.mp4", width=480)

## 第 6 章：拼接成片，找问题

`stitch_lap.py` 把 7 段按顺序接起来，接缝处做交叉淡化，叠加赛段标题和右下角的小地图（红点按每段的车速移动），再加上片头和片尾。

边看边记录：哪里转向和地图不一致？哪里画面跳变？小地图的红点和画面对得上吗？
把问题按“分镜、关键帧、提示词、拼接”四层归类，正好对应第 3 到 6 章。讲解提纲见 `docs/ch6_problems.md`。

In [ ]:
!"{PY}" "{PROJECT}/scripts/stitch_lap.py"
play(PROJECT / "output" / "dream_circuit_7shots.mp4", width=900)

## 第 7 章：用分镜修好 T7–T8 减速弯

地图上 T7 是右弯、T8 是左弯，但第 5 个镜头里只能看到一个左弯。下面按当时的顺序重现修复过程：

1. 原始镜头 5：K4 → K5 一个镜头跑完 T7–T8，模型直接跳过了 T7。
2. 只改提示词（第 5 章那一层）：T7 右弯仍然几乎看不出来。
3. 拆成两个镜头，T7 一个、T8 一个（第 3 章那一层）：但旧 K4 的远端是左弯，镜头开头先往左转。
4. 重画 K4，让远端只看到 T7 右弯（第 4 章那一层）：开头对了，但第一次画的 K5 同时看到右弯和左弯，画面在 1 秒内跳变。
5. 重画 K5，让它只看到 T8 左弯：问题解决。

两条规则：**一个镜头只改变一次方向；交界关键帧只画下一个弯。**

In [ ]:
play(MEDIA / "dream7_shot_05.mp4", MEDIA / "chicane_1_prompt_only.mp4", MEDIA / "chicane_2_split.mp4", width=360)
play(MEDIA / "chicane_3_new_K4.mp4", MEDIA / "dream8_shot_05.mp4", MEDIA / "dream8_shot_06.mp4", width=360)
show(PROJECT / "examples" / "keyframes" / "dream7_K4.png", PROJECT / "examples" / "keyframes" / "dream8_K4.png", width=360)
show(MEDIA / "chicane_K5_first_try.png", PROJECT / "examples" / "keyframes" / "dream8_K5.png", width=360)

8 镜头分镜表 `storyboards/lap_8shots.json` 只重做 K4、K5 两张关键帧和镜头 4–6，其余关键帧复用第 4 章、镜头复用第 5 章的结果。
路线 B：在浏览器里打开 `dream8_keyframes` 生成 K4、K5，对照分镜检查（不对就改场景描述再生成），运行下一格，刷新浏览器页面；再打开 `dream8_shot_04` 到 `dream8_shot_06` 生成视频，回来把下一格里视频的候选编号（`_t1_` 里的 1）改成你选的，再运行一次。
路线 A 直接运行下面几格。最后拼出 8 镜头成片，和第 6 章的版本对比。

In [ ]:
pick({k: f"{k}_*.png" for k in ("K4", "K5")}, COMFY / "output" / "dream8", COMFY / "input" / "dream8_{}.png")
pick({s: f"{s}_t1_*" for s in ("shot_04", "shot_05", "shot_06")}, COMFY / "output" / "video" / "dream8", COMFY / "output" / "video" / "dream8" / "selected" / "{}.mp4")

In [ ]:
for k in ("K4", "K5"):
    copy_missing(PROJECT / "examples" / "keyframes" / f"dream8_{k}.png", COMFY / "input" / f"dream8_{k}.png")
for i in (4, 5, 6):
    copy_missing(MEDIA / f"dream8_shot_0{i}.mp4", COMFY / "output" / "video" / "dream8" / "selected" / f"shot_0{i}.mp4")

In [ ]:
!"{PY}" "{PROJECT}/scripts/stitch_lap.py" --shots lap_8shots.json
play(PROJECT / "output" / "dream_circuit_8shots.mp4", width=900)

## 第 8 章：总结和练习

- 单个提示词适合一个动作，不适合一整圈；把任务拆成镜头，每个镜头只做一件事。
- 关键帧是镜头之间的“合同”：它画了什么，模型就会往哪里开。
- 出问题先归类（分镜、关键帧、提示词、拼接），再到对应的那一层修。

练习：成片里下坡直道和 T9 的衔接还有问题（直道没跑完就切进了 T9）。用同样的方法拆分镜头 7，重画交界关键帧，拼出你的版本。讲解提纲见 `docs/ch8_wrapup.md`。